# Setup

In [1]:
%load_ext autoreload
%autoreload 2
import logging
import os
import sys
import pandas as pd

# enforce more deterministic behavior in cuBLAS operations.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
# select a GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

sys.path.append("..")

from processor.core.interaction_conductor.llm_conductor import LLMConductor
from processor.core.ir_system.ir_data_model import RetrieverType
from processor.utils.logger import setup_logger
from processor.core.ir_system.ir_data_model import AbstractDocument
from processor.core.ir_system.ir_data_model import Table, TableContext
from processor.core.ir_system.ir_data_model import Knowledge, convert_multi_retriever_results_to_str
from processor.model.interface.model_factory import get_embed_model, get_llm
from processor.model.llm_message import Role
from processor.model.option import LLMOption
from processor.utils.json_processor import parse_json

from logging import Logger
from pandas import DataFrame

logger = setup_logger(
    name="processor_logger",
    log_path=os.path.join(".", "log"),
    level=logging.INFO,
    max_bytes=10_000_000,
    backup_count=5,
)

/home/luthfi/miniconda3/envs/pneuma/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# IR System

In [ ]:
# from processor.core.ir_system.lm_interface import LMInterface


# lm_interface = LMInterface({
#     "llm": get_llm("gpt-4o-mini")("gpt-4o-mini"),
#     "embed_model": get_embed_model()("model/weight/bge-base"),
# },
# logger)

In [ ]:
# output = lm_interface.retrieve(
#     RetrieverType.PNEUMA, "I need some shipping data.", ["buysite"], 3
# )

In [ ]:
# for i in output:
#     print(i.doc_id)

In [ ]:
# output2 = lm_interface.re_retrieve_with_feedback(
#     RetrieverType.PNEUMA, "Advanced shipping notice is relevant, but I need shipment numbers associated with them.", output, 3, ["buysite"]
# )

In [ ]:
# for i in output2:
#     print(i.doc_id)

# LLMConductor

## Definition

In [2]:
class Interaction:
    """
    Basically keeps track of every call to the process_input() function of LLMConductor
    """
    def __init__(self, human_input: str, llm_response: str) -> None:
        self.human_input = human_input
        self.llm_response = llm_response

    def __str__(self) -> str:
        return f"""{{"human input": {self.human_input}, "llm response": {self.llm_response}}}"""

class InformationNeedState:
    """
    Represents user's information need as a set of target schemas and SQLs to be executed over them.
    For example, if the user needs to know about the work addresses of faculty members, the target schemas
    may be ["name", "work address"], where name represents the names of the members, and work address represents
    the corresponding work address of each of them. After materialized by Materializer Engine, the SQLs can be
    executed sequentially over the materialized tables, and the outcome is useful to answer user's needs.
    """
    def __init__(self) -> None:
        self.target_schemas: dict[str, DataFrame] = dict()
        self.is_target_schemas_materialized = False
        self.column_descriptions: dict[str, dict[str, str]] = dict()
        self.sqls: list[str] = []

    def __str__(self) -> str:
        target_schemas_repr = ""
        for schema_id in self.target_schemas:
            table = self.target_schemas[schema_id]
            target_schemas_repr += f"\n- Table {schema_id}:\ncol: {" | ".join(list(table.columns))}"
            if len(table) > 0:
                # Sample 5 rows to represent the table
                sample_rows = table.sample(min(5, len(table)), random_state=42)
                sample_row_idx = 1
                for _, data in sample_rows.iterrows():
                    str_data = [str(i) for i in data]
                    target_schemas_repr += (
                        f"\n- sample row {sample_row_idx}: {" | ".join(str_data)}"
                    )
                    sample_row_idx += 1
            target_schemas_repr += "\n"
        return f"""Target schemas:
{target_schemas_repr.strip()}

Column descriptions of target schemas:
{self.column_descriptions}

SQLs to be run sequentially over the target schemas:
{self.sqls}"""

In [3]:
from processor.model.llm_message import LLMMessage


class ICPromptFactory:
    def get_sys_prompt(self, iteration_limit: int) -> str:
      return f"""Your role is to guide users in detecting, clarifying, and formalizing their possibly ambiguous information needs, eventually fulfilling them through structured data operations. You must converse and collaborate with users in evolving an Information Need State, which reflects their underlying information needs. This is a structured representation, consisting of:
    - `target_schemas` (dict[str, list[str]): A set of table schemas relevant to what users are looking for. The format is as follows: {{"Schema_ID_1": ["col_1", …], "Schema_ID_2": …, …}}. Each schema ID represents a conceptually coherent table. Each table is relevant to users' information needs. Target schemas, after finalized (i.e., confirmed with users), can be materialized by an external tool (more about this later).
    - `column_descriptions` (dict[str, dict[str, str]]): The descriptions of the columns of all target schemas. The format is as follows: {{"Schema_ID_1": {{"col_1": "This column represents …"}}, …}}
    - `sqls` (list[str]): A list of SQL queries over the (materialized) target schemas. Executing them (by an external tool) sequentially should produce relevant information to satisfy the underlying information needs of the users.
The end-to-and process is called a session, which is specific to a user. In each session, Information Need State starts empty but evolves over the course of the session. You must ensure the process is transparent and collaborative.
---

## Workflow
A session consists of multiple back-and-forth steps. In each step, you have at most {iteration_limit} iterations to select any of the following actions (mutually exclusive):
    - `internal_reasoning`: Reflect out loud (for yourself only).
    - `tool_call`: Call a tool to retrieve relevant information, evolve the state, etc.
    - `communicate_with_user`: Produce a user-facing message, which is either a summary of your actions in the step or a clarifying question.
Remember to close a step with `communicate_with_user`, so that they are aware of what has been done.
---

## AVAILABLE TOOLS
- **IR System**
    - Retrieves relevant tabular or textual data from our database based on natural-language prompts
    - For general inquiries, you may not need to use this tool and rely on your knowledge, but state clearly the sources of your information in the user-facing message.
    - Use when new or updated data is needed, but remember that calling this tool erases previously retrieved data (if any).
    - Args: `{{"prompt": "<retrieval query>"}}`

- **State Manipulation**
    - Updates Information Need State
    - Use when you have gathered enough signal to represent part of the user's needs formally. If user disagrees, iterate.
    - If the conversation has gone off-course, you can always reset the state (setting the values of target_schemas, column_descriptions, and sqls to be empty) and collaboratively rebuilding it with the user.
    - Users may inspect and give feedback on the current state at any time
    - Args: 
        {{
        "target_schemas": {{ "<id>": [<list of descriptive column names>] }},
        "column_descriptions": {{"<id>": {{ "<column name>": "<description of the column>" }} }}
        "sqls": [<list of SQL strings over target schema IDs>]
        }}

- **Materializer Engine**
    - Fills the current target schemas with actual data
    - Args: `""` (no input).
    - You may call this after finalizing the target schemas
    - Remember that if you reset/change the target schemas, then the materialization process has to be done again.

- **SQL Engine**
  - Runs the SQLs in the state over the materialized data
  - Args: `""` (no input).
  - You may call this after the target schemas have been materialized"""

    def get_env_state_prompt(
        self,
        curr_iteration: int,
        max_iteration: int,
        info_need_state: InformationNeedState,
        interaction_history: list[Interaction],
        actions_taken: list[str],
        curr_retrieval_results: dict[RetrieverType, list[AbstractDocument]],
        human_input: str,
    ) -> str:
        return f"""Relevant information for the current iteration in this step (iteration {curr_iteration} out of {max_iteration}):

INFORMATION NEED STATE:
{info_need_state}

ACTIONS YOU HAVE TAKEN FROM PREVIOUS ITERATIONS IN THIS STEP:
{actions_taken}

INTERACTION HISTORY (PAIRS OF HUMAN INPUT AND YOUR HUMAN-FACING RESPONSE):
{self.__convert_interactions_to_str(interaction_history)}

PREVIOUSLY RETRIEVED DATA FROM THE IR SYSTEM:
{convert_multi_retriever_results_to_str(curr_retrieval_results)}

CURRENT HUMAN INPUT:
{human_input}

Please output your decision for this step in the following format:
{{
    "intent": "communicate_with_user" | "internal_reasoning" | "tool_call",
    "message": null | "<string if intent is communicate_with_user or internal_reasoning>",
    "tool": null | "IR System" | "Materializer Engine" | "State Manipulation" | "SQL Engine",
    "args": null | { ... }
}}"""
    
    def get_direct_response_anyway_prompt(self) -> str:
        return """You have reached the iteration limit for this step. Please summarize the actions that you have done.
You are essentially asked to produce a `communicate_with_user` response but without the JSON format requirements. Simply output the summary."""   
    
    def __convert_interactions_to_str(self, interactions: list[Interaction]) -> str:
        interaction_repr = ""
        for interaction in interactions:
            interaction_repr += f"- {interaction}\n"
        interaction_repr = interaction_repr.strip()
        return interaction_repr

In [4]:
from processor.core.ir_system.lm_interface import LMInterface


ITERATION_LIMIT = 5
PAST_INTERACTIONS_LIMIT = 5


class LLMConductor:
    def __init__(self, llm_path: str, embed_path: str, logger: Logger) -> None:
        self.llm = get_llm(llm_path)(llm_path)
        self.embed_model = get_embed_model()(embed_path)
        self.logger = logger

        self.info_need_state = InformationNeedState()
        self.interaction_history: list[Interaction] = []

        self.prompt_factory = ICPromptFactory()
        self.current_retrieval_results: dict[RetrieverType, list[AbstractDocument]] = dict()

    def process_input(self, human_input: str) -> str:
        num_iteration = 0
        user_facing_response = ""
        is_user_facing_response = False
        llm_messages = [
            LLMMessage(
                role=Role.SYSTEM.value, content=self.prompt_factory.get_sys_prompt(ITERATION_LIMIT)
            )
        ]
        actions_taken: list[str] = []
        while not is_user_facing_response and num_iteration < ITERATION_LIMIT:
            num_iteration += 1
            llm_messages.append(
                LLMMessage(
                    role=Role.USER.value,
                    content=self.prompt_factory.get_env_state_prompt(
                        num_iteration,
                        ITERATION_LIMIT,
                        self.info_need_state,
                        self.interaction_history,
                        actions_taken,
                        self.current_retrieval_results,
                        human_input,
                    ),
                )
            )

            llm_output = self.llm.chat(llm_messages, LLMOption(json_mode=True))
            llm_messages.append(
                LLMMessage(role=Role.ASSISTANT.value, content=llm_output)
            )
            """Format of action:
            {
                "intent": "communicate_with_user" | "internal_reasoning" | "tool_call",
                "message": null | "<string>",
                "tool": null | "IR System" | "Materializer Engine" | "State Manipulation" | "SQL Engine",
                "args": null | { ... }
            }
            """
            action = parse_json(llm_output)
            intent: str = action.get("intent")
            action_message: None | str = action.get("message")
            tool: None | str = action.get("tool")
            args: None | dict = action.get("args")

            if intent == "communicate_with_user" and isinstance(action_message, str):
                self.interaction_history.append(
                    Interaction(human_input, action_message)
                )
                user_facing_response = action_message
                is_user_facing_response = True
            elif intent == "internal_reasoning" and isinstance(action_message, str):
                llm_messages.append(
                    LLMMessage(
                        role=Role.USER.value,
                        content=f"You did some internal reasoning: {action_message}",
                    )
                )
            elif intent == "tool_call" and tool is not None and args is not None:
                tool_outcome = self.__execute_tool(tool, args)
                llm_messages.append(
                    LLMMessage(role=Role.USER.value, content=tool_outcome)
                )

        if not is_user_facing_response:
            self.logger.info("Force produce user-facing response")
            llm_messages.append(
                LLMMessage(
                    role=Role.SYSTEM.value,
                    content=self.prompt_factory.get_direct_response_anyway_prompt(),
                )
            )
            user_facing_response = self.llm.chat(llm_messages)
            self.interaction_history.append(
                Interaction(human_input, user_facing_response)
            )
        return user_facing_response

    def __execute_tool(self, tool: str, args: str | dict) -> str:
        if tool == "IR System":
            self.logger.info(f"IR System request with params: {args}")
            ir_system = LMInterface(
                {"llm": self.llm, "embed_model": self.embed_model},
                self.logger,
            )
            self.current_retrieval_results = ir_system.retrieve_documents(
                args["prompt"],
                ["buysite"],
                5,  # Future-TODO: Change hard-coded sources and k
            )
            return "Successfully retrieved documents from the IR system. Notice that the `PREVIOUSLY RETRIEVED DATA FROM THE IR SYSTEM` has been updated."
        return "Tool calling failed."


## Working

In [5]:
llm_path = "model/weight/qwen3-8b"
embed_model_path = "model/weight/bge-base"
llm_conductor = LLMConductor(llm_path, embed_model_path, logger)

In [6]:
llm_conductor.process_input("I need some information about advanced shipping notices.")

Loading checkpoint shards: 100%|██████████| 5/5 [00:04<00:00,  1.05it/s]


QWEN: response: {
    "intent": "communicate_with_user",
    "message": "You mentioned you need information about advanced shipping notices. Could you clarify what specific details or aspects of advanced shipping notices you are interested in? For example, are you looking for shipping status updates, delivery timelines, tracking information, or something else related to logistics and supply chain management?",
    "tool": null,
    "args": null
}


'You mentioned you need information about advanced shipping notices. Could you clarify what specific details or aspects of advanced shipping notices you are interested in? For example, are you looking for shipping status updates, delivery timelines, tracking information, or something else related to logistics and supply chain management?'

In [7]:
llm_conductor.process_input("Yeah, I just want to know what the concept is all about. I just heard it somewhere.")

QWEN: response: {
    "intent": "communicate_with_user",
    "message": "Advanced shipping notices (ASNs) are detailed documents or messages sent by a supplier to a buyer before goods are shipped. They typically include information such as the items being shipped, quantities, packaging details, and expected delivery dates. ASNs help streamline the receiving process by providing visibility into what is coming and when. Is there anything specific you'd like to know about ASNs, such as their structure, benefits, or how they are used in logistics operations?",
    "tool": null,
    "args": null
}


"Advanced shipping notices (ASNs) are detailed documents or messages sent by a supplier to a buyer before goods are shipped. They typically include information such as the items being shipped, quantities, packaging details, and expected delivery dates. ASNs help streamline the receiving process by providing visibility into what is coming and when. Is there anything specific you'd like to know about ASNs, such as their structure, benefits, or how they are used in logistics operations?"

In [8]:
llm_conductor.process_input("Let's see. Do we have any data in our database related to ASNs?")

QWEN: response: {
    "intent": "tool_call",
    "message": "I will check our database to see if we have any relevant data related to advanced shipping notices (ASNs).",
    "tool": "IR System",
    "args": {"prompt": "advanced shipping notices data"}
}
[2025-07-24 15:46:59] INFO in 2236325732: IR System request with params: {'prompt': 'advanced shipping notices data'}
[2025-07-24 15:46:59] INFO in lm_interface: Starting document retrieval for prompt: advanced shipping notices data...
[2025-07-24 15:46:59] INFO in lm_interface: Selected retrievers: [<RetrieverType.PNEUMA: 'Pneuma'>, <RetrieverType.KNOWLEDGE_BASE: 'Knowledge Base'>]
[2025-07-24 15:46:59] INFO in lm_interface: => Retrieving from: RetrieverType.PNEUMA


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


[2025-07-24 15:47:00] INFO in lm_interface: => Initial retrieval returned 3 documents:
[2025-07-24 15:47:00] INFO in lm_interface: ==> Table ../../data_src/buysite/dataset/JI_ASN_CARRIER:
col: ASN_ID | ORG_ID | CARRIER | DOMAIN | SHIPMENT_CONTROL_ID | ELT_TS
sample row 1: 16596704 | 1962414 | Upsg - Ups Ground Service | companyName | 7473611 | 2023/11/12 02:00:32.266000000
sample row 2: 18987042 | 1962414 | Upsg - Ups Ground Service | companyName | 8719731 | 2024/05/24 16:45:32.237000000
sample row 3: 16717648 | 1962414 | Upsg - Ups Ground Service | companyName | 7540576 | 2023/11/21 08:00:31.761000000
sample row 4: 18898116 | 1962414 | FAST | companyName | 8672642 | 2024/05/18 00:15:30.626000000
sample row 5: 18898117 | 1962414 | FAST | companyName | 8672643 | 2024/05/18 00:15:30.626000000
[2025-07-24 15:47:00] INFO in lm_interface: ==> Table ../../data_src/buysite/dataset/JI_ASN:
col: ASN_ID | ASN_SHIPMENT_NUMBER | ORG_ID | SHIPMENT_DATE | DELIVERY_DATE | SHIPMENT_NOTES | ELT_TS
samp

'Yes, we have data related to advanced shipping notices (ASNs) in our database. The data includes tables such as `JI_ASN`, `JI_ASN_LINE`, and `JI_ASN_CARRIER`, which contain details about ASNs, shipment lines, and carrier information, respectively. Would you like me to formalize the target schemas and column descriptions for these tables to better understand the data structure?'

In [9]:
llm_conductor.process_input("Do you have any recommendations of what insights I can extract from these tables?")

QWEN: response: {
    "intent": "communicate_with_user",
    "message": "Based on the tables related to advanced shipping notices (ASNs), here are some insights you can extract:\n\n1. **ASN Overview**: From the `JI_ASN` table, you can get details about each ASN, including the shipment number, shipment date, delivery date, and any notes associated with the shipment.\n\n2. **Shipment Line Details**: The `JI_ASN_LINE` table provides information about the items shipped, including the quantity shipped, associated purchase order lines, and any comments.\n\n3. **Carrier Information**: The `JI_ASN_CARRIER` table contains details about the carrier used for each shipment, including the carrier name, shipment control ID, and the date the information was last updated.\n\nWould you like to explore any of these insights further, or is there a specific aspect of ASNs you are interested in? For example, you could look into the most common carriers used, the average delivery time, or the quantity of it

'Based on the tables related to advanced shipping notices (ASNs), here are some insights you can extract:\n\n1. **ASN Overview**: From the `JI_ASN` table, you can get details about each ASN, including the shipment number, shipment date, delivery date, and any notes associated with the shipment.\n\n2. **Shipment Line Details**: The `JI_ASN_LINE` table provides information about the items shipped, including the quantity shipped, associated purchase order lines, and any comments.\n\n3. **Carrier Information**: The `JI_ASN_CARRIER` table contains details about the carrier used for each shipment, including the carrier name, shipment control ID, and the date the information was last updated.\n\nWould you like to explore any of these insights further, or is there a specific aspect of ASNs you are interested in? For example, you could look into the most common carriers used, the average delivery time, or the quantity of items shipped per ASN.'

In [ ]:
llm_conductor.process_input("Do you have any recommendations of what insights I can extract from these tables?")